# CE-VAE + TUDA Output-Level Adversarial Adaptation

This notebook fine-tunes the CE-VAE model with **TUDA's output-level adversarial adaptation** — the most impactful TUDA component for improving paired-data scores.

**What this does:**
- Enables PatchGAN discriminator with **WGAN-GP** (from TUDA) on CE-VAE outputs
- Adds color consistency loss + gradient difference loss
- Fine-tunes from pre-trained CE-VAE checkpoint (epoch 119)

**Why this works (and why feature-level TUDA didn't):**
- Feature-level alignment hurts LSUI scores because it pulls encoder features toward a different domain (UIEB)
- Output-level adversarial training directly improves reconstruction quality: CE-VAE paper shows **+2.5 dB PSNR** from GAN loss alone (Fig. 5 ablation)
- The epoch119 checkpoint was trained WITHOUT adversarial loss — adding it is the single biggest improvement

**Requirements:** Kaggle T4 GPU or Colab T4, ~1.5-2 hours, LSUI dataset

**Steps:**
1. Setup environment
2. Link datasets
3. Train with output-level adversarial adaptation
4. Evaluate and compare baseline vs. enhanced

## 1. Environment Setup

In [ ]:
# Check GPU availability
!nvidia-smi
import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# Clone the repository and cd into it
# IMPORTANT: %cd MUST run — everything else depends on being inside the repo
!rm -rf ce-vae-tuda-underwater-enhancement 2>/dev/null
!git clone https://github.com/priyanshuharshbodhi1/ce-vae-tuda-underwater-enhancement.git
%cd ce-vae-tuda-underwater-enhancement

# Verify we're in the right directory
import os
assert os.path.exists('main.py'), "ERROR: main.py not found! %cd into the repo failed."
assert os.path.exists('src/models/cevae.py'), "ERROR: src/ not found!"
print(f"\nWorking directory: {os.getcwd()}")
print("Repository structure OK")

In [ ]:
# Install dependencies (wandb installed but set to offline — no login needed)
!pip install -q albumentations lightning omegaconf opencv-python-headless \
    'jsonargparse[signatures]>=4.27.7' Pillow pytorch-msssim torchmetrics wandb scikit-image termcolor lpips

# Disable wandb IMMEDIATELY — no login prompts, no online logging
import os
os.environ['WANDB_MODE'] = 'offline'
os.environ['WANDB_SILENT'] = 'true'
print("wandb set to offline mode (no login needed)")

## 2. Dataset Setup

You need:
1. **LSUI dataset** (paired: degraded + ground truth)
2. **Pre-trained CE-VAE checkpoint** (epoch 119)

**Note:** Unlike the TUDA feature-level notebook, we do NOT need unpaired real images (UIEB). Output-level adversarial training works purely on paired data.

In [ ]:
import os
import glob
from pathlib import Path

# Safety: ensure we're in the repo directory
if not os.path.exists('main.py'):
    for d in ['ce-vae-tuda-underwater-enhancement', '/kaggle/working/ce-vae-tuda-underwater-enhancement']:
        if os.path.exists(os.path.join(d, 'main.py')):
            os.chdir(d)
            print(f"Changed to: {os.getcwd()}")
            break

os.makedirs('data', exist_ok=True)

def link_path(src, dst):
    """Create symlink, skip if dst is a real dir/file."""
    if os.path.exists(dst):
        if os.path.islink(dst): os.unlink(dst)
        else: return  # real dir exists, skip
    os.symlink(src, dst)
    print(f'  Linked: {src} -> {dst}')

# ---- Detect environment ----
ON_KAGGLE = os.path.exists('/kaggle/input')
ON_COLAB = os.path.exists('/content')
search_root = '/kaggle/input' if ON_KAGGLE else ('/content' if ON_COLAB else '.')
print(f'Environment: {"Kaggle" if ON_KAGGLE else "Colab" if ON_COLAB else "Local"}')
print(f'Searching for datasets in {search_root}...\n')

# ---- 1. Search for LSUI paired dataset ----
print('[1/3] Looking for LSUI dataset (input + GT)...')
found_lsui = False

# Try: LSUI/input and LSUI/GT
lsui_dirs = glob.glob(f'{search_root}/**/LSUI', recursive=True)
if lsui_dirs:
    lsui_root = lsui_dirs[0]
    inp = os.path.join(lsui_root, 'input')
    gt = os.path.join(lsui_root, 'GT')
    if os.path.isdir(inp) and os.path.isdir(gt):
        link_path(inp, 'data/input')
        link_path(gt, 'data/GT')
        found_lsui = True

# Fallback: search for any 'input' and 'GT' folders
if not found_lsui:
    input_dirs = glob.glob(f'{search_root}/**/input', recursive=True)
    gt_dirs = glob.glob(f'{search_root}/**/GT', recursive=True)
    if input_dirs and gt_dirs:
        link_path(input_dirs[0], 'data/input')
        link_path(gt_dirs[0], 'data/GT')
        found_lsui = True

if found_lsui:
    n_inp = len(glob.glob('data/input/*.*'))
    n_gt = len(glob.glob('data/GT/*.*'))
    print(f'  Found LSUI: {n_inp} input, {n_gt} GT images')
else:
    print('  WARNING: LSUI dataset not found!')

# ---- 2. Search for checkpoint ----
print('\n[2/3] Looking for CE-VAE checkpoint (.ckpt)...')
ckpt_search = glob.glob(f'{search_root}/**/*.ckpt', recursive=True)
if ckpt_search:
    best_ckpt = ckpt_search[0]
    for c in ckpt_search:
        if 'epoch119' in c or 'cevae' in c.lower():
            best_ckpt = c
            break
    link_path(best_ckpt, 'data/lsui-cevae-epoch119.ckpt')
    print(f'  Found checkpoint: {best_ckpt}')
else:
    print('  WARNING: No .ckpt file found!')

# ---- 3. Generate train/val split file lists ----
print('\n[3/3] Generating train/val splits...')
if os.path.exists('data/input') and os.path.exists('data/GT'):
    inputs = sorted(glob.glob('data/input/**/*.*', recursive=True))
    targets = sorted(glob.glob('data/GT/**/*.*', recursive=True))
    img_exts = {'.png', '.jpg', '.jpeg', '.bmp', '.tiff'}
    inputs = [p for p in inputs if Path(p).suffix.lower() in img_exts]
    targets = [p for p in targets if Path(p).suffix.lower() in img_exts]
    
    if len(inputs) == len(targets) and len(inputs) > 0:
        split_idx = int(0.9 * len(inputs))
        with open('data/LSUI_train_input.txt', 'w') as f: f.write('\n'.join(inputs[:split_idx]))
        with open('data/LSUI_train_target.txt', 'w') as f: f.write('\n'.join(targets[:split_idx]))
        with open('data/LSUI_val_input.txt', 'w') as f: f.write('\n'.join(inputs[split_idx:]))
        with open('data/LSUI_val_target.txt', 'w') as f: f.write('\n'.join(targets[split_idx:]))
        print(f'  Train: {split_idx} pairs, Val: {len(inputs)-split_idx} pairs')
    elif len(inputs) != len(targets):
        print(f'  ERROR: input count ({len(inputs)}) != GT count ({len(targets)})')
    else:
        print(f'  ERROR: No images found in data/input/ or data/GT/')
else:
    print('  ERROR: data/input or data/GT not found')

print(f'\nCWD: {os.getcwd()}')
print('--- Data directory contents ---')
!ls -la data/

In [ ]:
# Verify everything is in place
required_files = [
    'data/lsui-cevae-epoch119.ckpt',
    'data/LSUI_train_input.txt',
    'data/LSUI_train_target.txt',
    'data/LSUI_val_input.txt',
    'data/LSUI_val_target.txt',
]

all_good = True
for f in required_files:
    exists = os.path.exists(f)
    status = 'OK' if exists else 'MISSING'
    print(f"  [{status}] {f}")
    if not exists:
        all_good = False

if all_good:
    print("\nAll files ready! Proceed to training.")
else:
    print("\nSome files are missing. Please upload them before training.")

## 3. Training

Fine-tune CE-VAE with TUDA output-level adversarial adaptation (WGAN-GP).

**Key changes from baseline:**
- `disc_enabled: True` with `disc_loss: wgan_gp` (TUDA-style stable adversarial training)
- `disc_start: 0` (start immediately since we fine-tune from a good checkpoint)
- `color_loss_weight: 0.5` (underwater color consistency)
- `gdl_loss_weight: 0.5` (edge/gradient detail preservation)

Expected time: **~1.5-2 hours on T4**

In [ ]:
# Ensure wandb is offline (main.py uses WandbLogger internally)
import os
os.environ['WANDB_MODE'] = 'offline'
os.environ['WANDB_SILENT'] = 'true'

# Safety: make sure we're in the repo directory
if not os.path.exists('main.py'):
    # Try to cd into the repo
    for d in ['ce-vae-tuda-underwater-enhancement', '/kaggle/working/ce-vae-tuda-underwater-enhancement']:
        if os.path.exists(os.path.join(d, 'main.py')):
            os.chdir(d)
            print(f"Changed to: {os.getcwd()}")
            break
    else:
        raise FileNotFoundError("Cannot find main.py! Re-run the 'Clone repository' cell above.")

# Start training with output-level adversarial adaptation!
# Expected: ~1.5-2 hours on T4 for 10 epochs
!python main.py -cfg configs/cevae_output_adapt_lsui.yaml --trainer.max_epochs 10

In [ ]:
# Find the best checkpoint
import glob
ckpt_files = sorted(glob.glob('training_logs/*/checkpoints/*.ckpt'))
print("Available checkpoints:")
if not ckpt_files:
    print("  No checkpoints found! Training may not have completed.")
    BEST_CKPT = None
else:
    for c in ckpt_files:
        size_mb = os.path.getsize(c) / (1024**2)
        print(f"  {c} ({size_mb:.1f} MB)")
    BEST_CKPT = [c for c in ckpt_files if 'last' in c][-1] if any('last' in c for c in ckpt_files) else ckpt_files[-1]
    print(f"\nUsing checkpoint: {BEST_CKPT}")

import sys, os

# Safety: ensure we're in the repo directory
if not os.path.exists('src/models/cevae.py'):
    for d in ['ce-vae-tuda-underwater-enhancement', '/kaggle/working/ce-vae-tuda-underwater-enhancement']:
        if os.path.exists(os.path.join(d, 'src/models/cevae.py')):
            os.chdir(d)
            break
sys.path.insert(0, os.getcwd())

import torch
import numpy as np
from omegaconf import OmegaConf
from src.build.from_config import instantiate_from_config
from src.metrics import compute as compute_metrics
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def load_model(config_path, ckpt_path):
    """Load a CE-VAE model from config and checkpoint."""
    config = OmegaConf.load(config_path)
    config.model.params.ckpt_path = None
    model = instantiate_from_config(config.model)
    sd = torch.load(ckpt_path, map_location='cpu', weights_only=False)['state_dict']
    model_keys = set(model.state_dict().keys())
    sd_filtered = {k: v for k, v in sd.items() if k in model_keys}
    model.load_state_dict(sd_filtered, strict=False)
    return model.eval().to(device)

if BEST_CKPT is None:
    print('Skipping model loading - no checkpoint found (training did not complete).')
    model_baseline = None
    model_adapted = None
else:
    print("Loading baseline CE-VAE (epoch 119, no adversarial loss)...")
    model_baseline = load_model('configs/cevae_E2E_lsui.yaml', 'data/lsui-cevae-epoch119.ckpt')
    print("Loading output-adapted CE-VAE (with WGAN-GP adversarial)...")
    model_adapted = load_model('configs/cevae_output_adapt_lsui.yaml', BEST_CKPT)
    print("Both models loaded!")

In [ ]:
import sys
sys.path.insert(0, '.')

import torch
import numpy as np
from omegaconf import OmegaConf
from src.build.from_config import instantiate_from_config
from src.metrics import compute as compute_metrics
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def load_model(config_path, ckpt_path):
    """Load a CE-VAE model from config and checkpoint."""
    config = OmegaConf.load(config_path)
    # Disable checkpoint loading in config (we load manually)
    config.model.params.ckpt_path = None
    model = instantiate_from_config(config.model)
    sd = torch.load(ckpt_path, map_location='cpu', weights_only=False)['state_dict']
    # Filter out discriminator keys for clean loading
    model_keys = set(model.state_dict().keys())
    sd_filtered = {k: v for k, v in sd.items() if k in model_keys}
    model.load_state_dict(sd_filtered, strict=False)
    return model.eval().to(device)

if BEST_CKPT is None:
    print('Skipping model loading - no checkpoint found (training did not complete).')
    model_baseline = None
    model_adapted = None
else:
    print("Loading baseline CE-VAE (epoch 119, no adversarial loss)...")
    model_baseline = load_model('configs/cevae_E2E_lsui.yaml', 'data/lsui-cevae-epoch119.ckpt')
    print("Loading output-adapted CE-VAE (with WGAN-GP adversarial)...")
    model_adapted = load_model('configs/cevae_output_adapt_lsui.yaml', BEST_CKPT)
    print("Both models loaded!")

In [ ]:
if model_baseline is None or model_adapted is None:
    print('Skipping evaluation - models were not loaded.')
else:
    from torch.utils.data import DataLoader
    from src.data.image_enhancement import DatasetTestFromImageFileList

    # Load test dataset
    test_dataset = DatasetTestFromImageFileList(
        size=256,
        test_images_list_file='data/LSUI_val_input.txt',
        test_target_images_list_file='data/LSUI_val_target.txt'
    )
    test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=2)

    def evaluate_model(model, loader, name):
        """Evaluate a model and return average metrics."""
        metrics_sum = {'psnr': 0, 'ssim': 0, 'uiqm': 0, 'uciqe': 0}
        count = 0

        with torch.no_grad():
            for batch in tqdm(loader, desc=f'Evaluating {name}'):
                x = batch['image'].permute(0, 3, 1, 2).float().to(device)
                y = batch['target'].permute(0, 3, 1, 2).float().to(device)

                xrec = model(x)

                y_np = torch.clamp(y, -1, 1).detach().cpu()
                y_np = ((y_np + 1) / 2 * 255).permute(0, 2, 3, 1).numpy().astype(np.uint8)

                xrec_np = torch.clamp(xrec, -1, 1).detach().cpu()
                xrec_np = ((xrec_np + 1) / 2 * 255).permute(0, 2, 3, 1).numpy().astype(np.uint8)

                for rec, gt in zip(xrec_np, y_np):
                    res = compute_metrics(rec, gt)
                    for k in metrics_sum:
                        metrics_sum[k] += res[k]
                    count += 1

        return {k: v / count for k, v in metrics_sum.items()}

    print("Evaluating baseline CE-VAE (no adversarial loss)...")
    baseline_metrics = evaluate_model(model_baseline, test_loader, 'Baseline')

    print("\nEvaluating output-adapted CE-VAE (WGAN-GP adversarial)...")
    adapted_metrics = evaluate_model(model_adapted, test_loader, 'Output-Adapted')

    # Print comparison
    print("\n" + "="*65)
    print("RESULTS COMPARISON: Baseline vs Output-Adapted (TUDA WGAN-GP)")
    print("="*65)
    header = f"{'Metric':<10} {'Baseline (no GAN)':>18} {'+ Output Adapt':>15} {'Delta':>10}"
    print(header)
    print("-"*55)

    results_txt = "="*65 + "\nRESULTS COMPARISON\n" + "="*65 + "\n" + header + "\n" + "-"*55 + "\n"

    for k in ['psnr', 'ssim', 'uiqm', 'uciqe']:
        delta = adapted_metrics[k] - baseline_metrics[k]
        arrow = '+' if delta > 0 else ''
        line = f"{k.upper():<10} {baseline_metrics[k]:>18.4f} {adapted_metrics[k]:>15.4f} {arrow}{delta:>9.4f}"
        print(line)
        results_txt += line + "\n"

    print("="*65)
    results_txt += "="*65 + "\n"

    with open('results_comparison.txt', 'w') as f:
        f.write(results_txt)
    print(f"\nResults saved to results_comparison.txt")

In [ ]:
if model_baseline is None or model_adapted is None:
    print('Skipping visual comparison - models were not loaded.')
else:
    import matplotlib.pyplot as plt

    # Get a few samples for visual comparison
    sample_batch = next(iter(DataLoader(test_dataset, batch_size=4, shuffle=True)))
    x = sample_batch['image'].permute(0, 3, 1, 2).float().to(device)
    y = sample_batch['target'].permute(0, 3, 1, 2).float().to(device)

    with torch.no_grad():
        rec_baseline = model_baseline(x)
        rec_adapted = model_adapted(x)

    def tensor_to_img(t):
        t = torch.clamp(t, -1, 1)
        return ((t + 1) / 2).cpu().permute(1, 2, 0).numpy()

    fig, axes = plt.subplots(4, 4, figsize=(20, 20))
    titles = ['Input (Degraded)', 'Baseline CE-VAE', 'CE-VAE + Output Adapt', 'Ground Truth']
    for i in range(4):
        imgs = [x[i], rec_baseline[i], rec_adapted[i], y[i]]
        for j, (img, title) in enumerate(zip(imgs, titles)):
            axes[i, j].imshow(tensor_to_img(img))
            if i == 0:
                axes[i, j].set_title(title, fontsize=14, fontweight='bold')
            axes[i, j].axis('off')

    plt.suptitle('CE-VAE Baseline vs. Output-Adapted (TUDA WGAN-GP)', fontsize=16, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig('comparison_results.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Comparison saved to comparison_results.png")

In [ ]:
if BEST_CKPT is not None:
    import shutil
    output_ckpt = 'cevae_output_adapted.ckpt'
    shutil.copy(BEST_CKPT, output_ckpt)
    print(f"Final checkpoint saved as: {output_ckpt}")
    print(f"Size: {os.path.getsize(output_ckpt) / (1024**2):.1f} MB")
else:
    print('No checkpoint to save.')

## Why This Works

### What we integrated from TUDA:
**Output-level adversarial adaptation with WGAN-GP** — the single most impactful component.

### Why feature-level TUDA failed (previous attempt):
- Feature-level alignment aligns encoder features between LSUI (paired) and UIEB (real unpaired)
- This pulls encoder features AWAY from optimal LSUI reconstruction
- Result: scores dropped because the model was optimized for domain invariance, not reconstruction quality

### Why output-level adaptation works:
1. **PatchGAN discriminator** forces the network to generate realistic-looking outputs (no blurriness, proper textures)
2. **WGAN-GP** (from TUDA) provides stable gradients — no mode collapse, no training instability
3. **Adaptive weighting** automatically balances reconstruction vs. adversarial loss
4. **Color loss** improves underwater color correction (addresses color cast)
5. **GDL loss** preserves edge details and fine structures

### Expected improvements:
- **PSNR**: +1.5 to +2.5 dB (the GAN loss concentrates PSNR distribution, from CE-VAE paper Fig. 5)
- **SSIM**: +0.01 to +0.03 (sharper edges, better structure)
- **LPIPS**: -0.02 to -0.05 (better perceptual quality)
- **UIQM/UCIQE**: should improve due to better color and contrast